# Step 11 — create alternative BEAM partitions for manually selected blocks

**# of cells in notebook:** 1

**Purpose:** Create alternative block subdivisions for source blocks that were selected for additional review after Step 10. Building-level tessellation cells are partitioned into two, three, and four spatially contiguous groups using a bounded beam-search algorithm. The search favors simple internal boundaries while maintaining specified limits on the share of total building footprint area assigned to each partition.

**Input:**

- a manually specified list of source blocks selected after reviewing the Step 10 diagnostic table and visually inspecting the blocks
- `building_level_tessellation.gpkg` from Step 3
- `building_context_features_new_dist.gpkg` from Step 4
- `context_bldg_id` — the stable building/tessellation identifier
- `area_m_utm` — building footprint area from Step 4; used as the BEAM partition-balancing weight
- `cell_area_m2` — tessellation-cell area from Step 4; carried into the BEAM outputs and summed for each candidate partition

**Output:**

Within each selected block folder:

- `graph_boundary_penalty_partitions_beam.gpkg`
- candidate cell and dissolved-partition layers:
  - `selected_cells_n2`
  - `selected_dissolved_n2`
  - `selected_cells_n3`
  - `selected_dissolved_n3`
  - `selected_cells_n4`
  - `selected_dissolved_n4`

At the base block directory:

- `graph_boundary_penalty_partitions_beam_summary.csv`
- `beam_blocks.csv` — one-row-per-parent inventory of blocks manually routed through the BEAM workflow

**Main logic:**

**Cell 1 — Create contiguous BEAM partitions**

1. Reads the Step 3 building-level tessellation and joins `area_m_utm` and `cell_area_m2` from the Step 4 building-context layer using `context_bldg_id`.
2. Builds a tessellation adjacency graph based on shared polygon boundaries.
3. Searches for candidate subdivisions into 2, 3, and 4 contiguous parts using bounded beam search.
4. Requires every final partition to be graph-contiguous.
5. Constrains each partition's share of total building footprint area:
   - for n = 2, each part must contain 40–60%;
   - for n = 3, each part must contain 18–45%;
   - for n = 4, each part must contain 18–45%.
6. Scores candidate partitions primarily according to internal-boundary simplicity, with penalties for longer boundaries, additional cut edges, diagonal boundaries, and building-area imbalance.
7. Dissolves the selected tessellation groups and records both summed building footprint area (`building_area_sum`) and summed tessellation-cell area (`cell_area_m2`) for each candidate new block.
8. Writes the selected cell assignments, dissolved candidate partitions, diagnostic summary information, and the BEAM-parent inventory.


In [ ]:
r"""
graph_boundary_penalty_partitions_beam.py

Purpose
-------
For manually selected source-block folders, partition building-level tessellation
cells into 2, 3, and 4 contiguous groups using bounded BEAM SEARCH.

This is a faster replacement for the exhaustive recursive graph-partition search.
It keeps only the best partial partition states at each depth, so 4-part searches
should run in minutes rather than hours on the test blocks.

Inputs per block
----------------
building_level_tessellation.gpkg (Step 3) + building_context_features_new_dist.gpkg (Step 4)

Expected Step 4 fields
----------------------
area_m_utm   = building footprint area; BEAM balancing weight
cell_area_m2 = tessellation-cell area; summed for each candidate partition

Outputs per block
-----------------
graph_boundary_penalty_partitions_beam.gpkg
    selected_cells_n2
    selected_dissolved_n2
    selected_cells_n3
    selected_dissolved_n3
    selected_cells_n4
    selected_dissolved_n4

Summary CSV
-----------
E:\_johannesburg\_analysis\heterogeneous_largePop_blocks\graph_boundary_penalty_partitions_beam_summary.csv

Core rules
----------
1. Every final partition must be graph-contiguous.
2. Multipart/disconnected final partitions are not accepted.
3. Boundary simplicity is prioritized over perfectly equal building area.
4. Weight-share constraints:
   - n = 2: each final group must contain 40% to 60% of total building area.
   - n = 3: each final group must contain 18% to 45% of total building area.
   - n = 4: each final group must contain 18% to 45% of total building area.
"""

from pathlib import Path
import math
import warnings
from dataclasses import dataclass

import geopandas as gpd
import pandas as pd
import networkx as nx
from shapely.ops import unary_union
from shapely.geometry import LineString, MultiLineString, GeometryCollection


# =============================================================================
# USER SETTINGS
# =============================================================================

root_dir = Path(r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks")

block_folders_to_process = [
    "_52730",
    "_56219",
    "_62733",
    "_71608",
    "_72054",
    "_76290",
    "_83562",
    "_84830",
    "_89704"
]

# To test just one block, use something like:
# block_folders_to_process = ["_2740"]

n_parts_to_process = [2, 3, 4]
# To test just 4-part partitions:
# n_parts_to_process = [4]

tessellation_gpkg_name = "building_level_tessellation.gpkg"
tessellation_layer = "building_level_tessellation"

context_gpkg_name = "building_context_features_new_dist.gpkg"
context_layer = "building_context_features_new_dist"

output_gpkg_name = "graph_boundary_penalty_partitions_beam.gpkg"
summary_csv = root_dir / "graph_boundary_penalty_partitions_beam_summary.csv"
beam_inventory_csv = root_dir / "beam_blocks.csv"

context_id_field = "context_bldg_id"
weight_field = "area_m_utm"
cell_area_field = "cell_area_m2"

OVERWRITE_OUTPUT = True

# Final partition share constraints, expressed as share of total building area.
FINAL_SHARE_BOUNDS = {
    2: (0.40, 0.60),
    3: (0.18, 0.45),
    4: (0.18, 0.45),
}

# Main runtime/quality knobs.
# Larger values search more thoroughly but run slower.
BEAM_WIDTH = 12
MAX_SPLIT_CANDIDATES_PER_STEP = 40

# Optional hard cap on total expanded states per n_parts search.
# Set to None to disable.
MAX_TOTAL_EXPANSIONS = 20000

# Candidate generation angles, in degrees.
# Actual scoring penalizes diagonal boundaries, so diagonal-ish ordering
# directions are allowed but discouraged if they produce diagonal boundaries.
CANDIDATE_ANGLES_DEGREES = [
    0, 90,
    15, 75, 105, 165,
    30, 60, 120, 150,
    45, 135,
]

# Score weights. Boundary score dominates.
BOUNDARY_LENGTH_WEIGHT = 1.0
CUT_EDGE_WEIGHT = 2.0
DIAGONAL_ORIENTATION_WEIGHT = 25.0

# Balance is intentionally light because share constraints already enforce bounds.
BALANCE_WEIGHT = 0.02

# Partial-state balance weight nudges groups allocated k final parts toward
# k / n_parts of total building area. It is deliberately light.
PARTIAL_ALLOCATION_BALANCE_WEIGHT = 0.01

# Numerical tolerances.
SHARED_LENGTH_EPS = 1e-8
SHARE_TOLERANCE = 1e-9


# =============================================================================
# GEOMETRY / GRAPH HELPERS
# =============================================================================

def clean_geometry(gdf):
    """Basic geometry cleanup."""
    out = gdf.copy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out["geometry"] = out.geometry.buffer(0)
    return out


def line_segment_orientation_penalty(geom):
    """
    Return diagonal-weighted boundary length.

    Horizontal and vertical segments receive near-zero penalty.
    Diagonal segments receive larger penalty.
    """
    if geom is None or geom.is_empty:
        return 0.0

    total = 0.0

    def process_linestring(ls):
        subtotal = 0.0
        coords = list(ls.coords)
        for i in range(len(coords) - 1):
            x1, y1 = coords[i]
            x2, y2 = coords[i + 1]
            dx = x2 - x1
            dy = y2 - y1
            length = math.hypot(dx, dy)
            if length <= 0:
                continue
            sin_abs = abs(dy) / length
            cos_abs = abs(dx) / length
            subtotal += length * min(sin_abs, cos_abs)
        return subtotal

    if isinstance(geom, LineString):
        total += process_linestring(geom)
    elif isinstance(geom, MultiLineString):
        for part in geom.geoms:
            total += process_linestring(part)
    elif isinstance(geom, GeometryCollection):
        for part in geom.geoms:
            if isinstance(part, LineString):
                total += process_linestring(part)
            elif isinstance(part, MultiLineString):
                for subpart in part.geoms:
                    total += process_linestring(subpart)

    return total


def build_adjacency_graph_and_edges(gdf):
    """
    Build adjacency graph using shared boundary segments.

    Nodes are tessellation cells.
    Edges connect cells that share a non-trivial boundary segment.

    edge_records contains:
        i, j, shared_len, diagonal_len
    """
    gdf = gdf.reset_index(drop=True).copy()

    graph = nx.Graph()
    graph.add_nodes_from(gdf.index.tolist())

    edge_records = []
    sindex = gdf.sindex

    for i, geom_i in enumerate(gdf.geometry):
        candidate_idxs = list(sindex.intersection(geom_i.bounds))

        for j in candidate_idxs:
            if j <= i:
                continue

            geom_j = gdf.geometry.iloc[j]

            if not geom_i.intersects(geom_j):
                continue

            inter = geom_i.boundary.intersection(geom_j.boundary)
            shared_len = inter.length if not inter.is_empty else 0.0

            if shared_len > SHARED_LENGTH_EPS:
                diagonal_len = line_segment_orientation_penalty(inter)
                graph.add_edge(i, j, shared_len=shared_len, diagonal_len=diagonal_len)
                edge_records.append((i, j, shared_len, diagonal_len))

    return graph, edge_records


def connected_subset(graph, ids):
    """Return True if ids form one connected graph component."""
    ids = list(ids)
    if len(ids) <= 1:
        return True
    return nx.is_connected(graph.subgraph(ids))


def component_count(graph, ids):
    """Number of graph components in ids."""
    ids = list(ids)
    if not ids:
        return 0
    if len(ids) == 1:
        return 1
    return nx.number_connected_components(graph.subgraph(ids))


def add_projection_values(gdf, angle_radians):
    """Project cell centroids onto an axis."""
    ux = math.cos(angle_radians)
    uy = math.sin(angle_radians)
    centroids = gdf.geometry.centroid
    return centroids.x * ux + centroids.y * uy


def get_mrr_angles_degrees(gdf):
    """Return long-axis and short-axis angles from the minimum rotated rectangle."""
    try:
        geom = unary_union(gdf.geometry)
        mrr = geom.minimum_rotated_rectangle
        coords = list(mrr.exterior.coords)

        edges = []
        for i in range(len(coords) - 1):
            x1, y1 = coords[i]
            x2, y2 = coords[i + 1]
            dx = x2 - x1
            dy = y2 - y1
            length = math.hypot(dx, dy)
            if length > 0:
                angle = math.degrees(math.atan2(dy, dx)) % 180
                edges.append((length, angle))

        if not edges:
            return []

        long_angle = max(edges, key=lambda x: x[0])[1]
        short_angle = (long_angle + 90) % 180
        return [long_angle, short_angle]

    except Exception:
        return []


# =============================================================================
# SCORING HELPERS
# =============================================================================

def boundary_score_between_sets(left_ids, right_ids, edge_records):
    """Score the actual internal boundary between two cell sets."""
    left = set(left_ids)
    right = set(right_ids)

    boundary_length = 0.0
    diagonal_len = 0.0
    cut_edges = 0

    for i, j, shared_len, diag_len in edge_records:
        if (i in left and j in right) or (i in right and j in left):
            boundary_length += shared_len
            diagonal_len += diag_len
            cut_edges += 1

    score = (
        BOUNDARY_LENGTH_WEIGHT * boundary_length
        + CUT_EDGE_WEIGHT * cut_edges
        + DIAGONAL_ORIENTATION_WEIGHT * diagonal_len
    )

    return {
        "score": score,
        "boundary_length": boundary_length,
        "diagonal_len": diagonal_len,
        "cut_edges": cut_edges,
    }


def final_internal_boundary_score(groups, edge_records):
    """Score all internal boundaries among current or final groups."""
    node_to_group = {}
    for gid, ids in enumerate(groups, start=1):
        for node in ids:
            node_to_group[node] = gid

    boundary_length = 0.0
    diagonal_len = 0.0
    cut_edges = 0

    for i, j, shared_len, diag_len in edge_records:
        if node_to_group.get(i) != node_to_group.get(j):
            boundary_length += shared_len
            diagonal_len += diag_len
            cut_edges += 1

    score = (
        BOUNDARY_LENGTH_WEIGHT * boundary_length
        + CUT_EDGE_WEIGHT * cut_edges
        + DIAGONAL_ORIENTATION_WEIGHT * diagonal_len
    )

    return {
        "score": score,
        "boundary_length": boundary_length,
        "diagonal_len": diagonal_len,
        "cut_edges": cut_edges,
    }


def balance_penalty_for_final_groups(gdf, groups, total_weight, n_parts):
    """Light penalty for imbalance inside the allowed ranges."""
    ideal = total_weight / n_parts
    penalty = 0.0
    for ids in groups:
        w = float(gdf.loc[list(ids), weight_field].sum())
        penalty += abs(w - ideal)
    return penalty * BALANCE_WEIGHT


def partial_allocation_balance_penalty(gdf, specs, total_weight, n_parts):
    """
    Light penalty for partial states.

    A group allocated k final parts is nudged toward k/n_parts of total area.
    """
    penalty = 0.0
    ideal_unit = total_weight / n_parts

    for ids, k_alloc in specs:
        w = float(gdf.loc[list(ids), weight_field].sum())
        ideal = ideal_unit * k_alloc
        penalty += abs(w - ideal)

    return penalty * PARTIAL_ALLOCATION_BALANCE_WEIGHT


def partial_state_score(gdf, specs, edge_records, total_weight, n_parts):
    """Score a partial beam state."""
    current_groups = [ids for ids, _ in specs]
    bs = final_internal_boundary_score(current_groups, edge_records)
    bal = partial_allocation_balance_penalty(gdf, specs, total_weight, n_parts)
    return {
        "score": bs["score"] + bal,
        "boundary_score": bs["score"],
        "balance_penalty": bal,
        "boundary_length": bs["boundary_length"],
        "diagonal_len": bs["diagonal_len"],
        "cut_edges": bs["cut_edges"],
    }


# =============================================================================
# CANDIDATE SPLIT GENERATION
# =============================================================================

def share_bounds_for_subset(total_weight, subset_weight, n_parts, k_left, k_right):
    """
    Absolute weight bounds for the left side of a split.

    Each final group must be within FINAL_SHARE_BOUNDS[n_parts] as a share
    of the TOTAL block building area.
    """
    min_share, max_share = FINAL_SHARE_BOUNDS[n_parts]

    left_min = k_left * min_share * total_weight
    left_max = k_left * max_share * total_weight

    right_min = k_right * min_share * total_weight
    right_max = k_right * max_share * total_weight

    left_min_from_right = subset_weight - right_max
    left_max_from_right = subset_weight - right_min

    allowed_min = max(left_min, left_min_from_right)
    allowed_max = min(left_max, left_max_from_right)

    return allowed_min, allowed_max


def spec_feasible_by_weight(gdf, ids, total_weight, n_parts, k_alloc):
    """
    Check whether a partial group with k_alloc future final groups has
    enough and not too much weight to eventually satisfy final share bounds.
    """
    min_share, max_share = FINAL_SHARE_BOUNDS[n_parts]
    w = float(gdf.loc[list(ids), weight_field].sum())
    min_w = k_alloc * min_share * total_weight
    max_w = k_alloc * max_share * total_weight

    return (
        w >= min_w - SHARE_TOLERANCE
        and w <= max_w + SHARE_TOLERANCE
    )


def generate_connected_split_candidates(
    gdf,
    graph,
    edge_records,
    ids,
    total_weight,
    n_parts,
    k_left,
    k_right,
    max_candidates=40,
):
    """
    Generate connected left/right split candidates for a current group.

    For each projection cut, this tests BOTH orientations:
        prefix -> left, suffix -> right
        suffix -> left, prefix -> right
    """
    ids = list(ids)

    if len(ids) < 2:
        return []

    subset_weight = float(gdf.loc[ids, weight_field].sum())

    allowed_min, allowed_max = share_bounds_for_subset(
        total_weight=total_weight,
        subset_weight=subset_weight,
        n_parts=n_parts,
        k_left=k_left,
        k_right=k_right,
    )

    allowed_min = max(allowed_min, 0.0)
    allowed_max = min(allowed_max, subset_weight)

    if allowed_min > allowed_max:
        return []

    sub = gdf.loc[ids].copy()

    angle_degrees = list(CANDIDATE_ANGLES_DEGREES)
    angle_degrees.extend(get_mrr_angles_degrees(sub))
    angle_degrees = sorted({round(a % 180, 6) for a in angle_degrees})

    candidates = []
    seen = set()

    def try_add_candidate(left_ids_raw, right_ids_raw, left_weight, right_weight, angle_deg):
        if not left_ids_raw or not right_ids_raw:
            return

        if left_weight < allowed_min - SHARE_TOLERANCE:
            return

        if left_weight > allowed_max + SHARE_TOLERANCE:
            return

        left_ids = tuple(sorted(left_ids_raw))
        right_ids = tuple(sorted(right_ids_raw))

        # If k_left == k_right, mirrored split is equivalent.
        # If k_left != k_right, mirrored split is not equivalent.
        if k_left == k_right:
            candidate_key = min(left_ids, right_ids)
        else:
            candidate_key = (left_ids, k_left)

        if candidate_key in seen:
            return
        seen.add(candidate_key)

        if not connected_subset(graph, left_ids):
            return
        if not connected_subset(graph, right_ids):
            return

        bs = boundary_score_between_sets(left_ids, right_ids, edge_records)

        desired_left = (allowed_min + allowed_max) / 2.0
        balance_penalty = abs(left_weight - desired_left) * BALANCE_WEIGHT

        total_score = bs["score"] + balance_penalty

        candidates.append({
            "score": total_score,
            "split_score": bs["score"],
            "balance_penalty": balance_penalty,
            "left_ids": left_ids,
            "right_ids": right_ids,
            "left_weight": left_weight,
            "right_weight": right_weight,
            "angle_deg": angle_deg,
            "boundary_length": bs["boundary_length"],
            "diagonal_len": bs["diagonal_len"],
            "cut_edges": bs["cut_edges"],
        })

    for angle_deg in angle_degrees:
        angle_rad = math.radians(angle_deg)
        projections = add_projection_values(sub, angle_rad)

        ordered_ids = [
            x for _, x in sorted(zip(projections.values, sub.index.tolist()))
        ]

        running_weight = 0.0

        for cut_idx in range(1, len(ordered_ids)):
            moved_id = ordered_ids[cut_idx - 1]
            running_weight += float(gdf.at[moved_id, weight_field])

            prefix_ids = ordered_ids[:cut_idx]
            suffix_ids = ordered_ids[cut_idx:]

            prefix_weight = running_weight
            suffix_weight = subset_weight - prefix_weight

            try_add_candidate(
                left_ids_raw=prefix_ids,
                right_ids_raw=suffix_ids,
                left_weight=prefix_weight,
                right_weight=suffix_weight,
                angle_deg=angle_deg,
            )

            try_add_candidate(
                left_ids_raw=suffix_ids,
                right_ids_raw=prefix_ids,
                left_weight=suffix_weight,
                right_weight=prefix_weight,
                angle_deg=angle_deg,
            )

    candidates.sort(key=lambda d: d["score"])
    return candidates[:max_candidates]


# =============================================================================
# BEAM SEARCH
# =============================================================================

@dataclass(frozen=True)
class BeamState:
    """
    specs:
        Tuple of (ids_tuple, k_alloc)

    Each spec is a current connected group and the number of final parts
    it must eventually produce.
    """
    specs: tuple
    score: float
    boundary_score: float
    balance_penalty: float
    boundary_length: float
    diagonal_len: float
    cut_edges: int


def canonicalize_specs(specs):
    """Normalize state representation so duplicate states can be removed."""
    norm = []
    for ids, k_alloc in specs:
        norm.append((tuple(sorted(ids)), int(k_alloc)))
    norm.sort(key=lambda x: (x[1], x[0]))
    return tuple(norm)


def state_from_specs(gdf, specs, edge_records, total_weight, n_parts):
    """Create BeamState from specs."""
    specs = canonicalize_specs(specs)
    sc = partial_state_score(gdf, specs, edge_records, total_weight, n_parts)

    return BeamState(
        specs=specs,
        score=float(sc["score"]),
        boundary_score=float(sc["boundary_score"]),
        balance_penalty=float(sc["balance_penalty"]),
        boundary_length=float(sc["boundary_length"]),
        diagonal_len=float(sc["diagonal_len"]),
        cut_edges=int(sc["cut_edges"]),
    )


def final_solution_from_state(gdf, state, edge_records, total_weight, n_parts):
    """Convert a finished beam state into a solution dict."""
    groups = [ids for ids, k_alloc in state.specs]

    min_share, max_share = FINAL_SHARE_BOUNDS[n_parts]
    for ids in groups:
        w = float(gdf.loc[list(ids), weight_field].sum())
        share = w / total_weight if total_weight else 0.0
        if share < min_share - SHARE_TOLERANCE or share > max_share + SHARE_TOLERANCE:
            return None

    bs = final_internal_boundary_score(groups, edge_records)
    bal = balance_penalty_for_final_groups(gdf, groups, total_weight, n_parts)

    return {
        "groups": groups,
        "score": bs["score"] + bal,
        "boundary_score": bs["score"],
        "balance_penalty": bal,
        "boundary_length": bs["boundary_length"],
        "diagonal_len": bs["diagonal_len"],
        "cut_edges": bs["cut_edges"],
    }


def find_best_partition_beam(gdf, graph, edge_records, n_parts):
    """Find a contiguous k-way partition using bounded beam search."""
    total_weight = float(gdf[weight_field].sum())
    all_ids = tuple(sorted(gdf.index.tolist()))

    initial_state = state_from_specs(
        gdf=gdf,
        specs=((all_ids, n_parts),),
        edge_records=edge_records,
        total_weight=total_weight,
        n_parts=n_parts,
    )

    beam = [initial_state]
    total_expansions = 0

    print(
        f"Beam search settings: BEAM_WIDTH={BEAM_WIDTH}, "
        f"MAX_SPLIT_CANDIDATES_PER_STEP={MAX_SPLIT_CANDIDATES_PER_STEP}"
    )

    # Each expansion depth increases current group count by 1.
    for depth in range(1, n_parts):
        next_states_by_key = {}

        print(f"  Beam depth {depth}/{n_parts - 1}; current beam size={len(beam)}")

        stop_now = False

        for state in beam:
            specs = list(state.specs)

            for split_index, (ids_to_split, k_alloc) in enumerate(specs):
                if k_alloc <= 1:
                    continue

                for k_left in range(1, k_alloc):
                    k_right = k_alloc - k_left
                    if k_left > k_right:
                        continue

                    candidates = generate_connected_split_candidates(
                        gdf=gdf,
                        graph=graph,
                        edge_records=edge_records,
                        ids=ids_to_split,
                        total_weight=total_weight,
                        n_parts=n_parts,
                        k_left=k_left,
                        k_right=k_right,
                        max_candidates=MAX_SPLIT_CANDIDATES_PER_STEP,
                    )

                    for cand in candidates:
                        total_expansions += 1

                        if MAX_TOTAL_EXPANSIONS is not None and total_expansions > MAX_TOTAL_EXPANSIONS:
                            print(f"  Reached MAX_TOTAL_EXPANSIONS={MAX_TOTAL_EXPANSIONS}; stopping expansion.")
                            stop_now = True
                            break

                        left_ids = cand["left_ids"]
                        right_ids = cand["right_ids"]

                        if not spec_feasible_by_weight(gdf, left_ids, total_weight, n_parts, k_left):
                            continue
                        if not spec_feasible_by_weight(gdf, right_ids, total_weight, n_parts, k_right):
                            continue

                        new_specs = specs.copy()
                        new_specs.pop(split_index)
                        new_specs.append((left_ids, k_left))
                        new_specs.append((right_ids, k_right))

                        new_state = state_from_specs(
                            gdf=gdf,
                            specs=tuple(new_specs),
                            edge_records=edge_records,
                            total_weight=total_weight,
                            n_parts=n_parts,
                        )

                        key = new_state.specs
                        existing = next_states_by_key.get(key)
                        if existing is None or new_state.score < existing.score:
                            next_states_by_key[key] = new_state

                    if stop_now:
                        break
                if stop_now:
                    break
            if stop_now:
                break

        next_states = sorted(next_states_by_key.values(), key=lambda s: s.score)
        beam = next_states[:BEAM_WIDTH]

        print(f"    candidate states after expansion: {len(next_states):,}")
        print(f"    retained beam states: {len(beam):,}")
        if beam:
            print(f"    best partial score: {beam[0].score:,.2f}")

        if not beam:
            return None

        if stop_now:
            break

    # Select best finished state.
    finished = []
    for state in beam:
        if all(k_alloc == 1 for _, k_alloc in state.specs):
            solution = final_solution_from_state(
                gdf=gdf,
                state=state,
                edge_records=edge_records,
                total_weight=total_weight,
                n_parts=n_parts,
            )
            if solution is not None:
                finished.append(solution)

    if not finished:
        return None

    finished.sort(key=lambda s: s["score"])
    return finished[0]


# =============================================================================
# OUTPUT HELPERS
# =============================================================================

def assign_group_ids(gdf, groups, part_field):
    """Assign 1..k IDs to the final groups."""
    out = pd.Series(index=gdf.index, dtype="Int64")
    for gid, ids in enumerate(groups, start=1):
        out.loc[list(ids)] = gid
    return out


def make_dissolved_layer(cells_gdf, part_field, block_id, n_parts, solution, graph):
    """Dissolve cells and add diagnostics."""
    cells_gdf = cells_gdf.copy()
    total_weight = float(cells_gdf[weight_field].sum())
    target_weight = total_weight / n_parts

    cell_counts = cells_gdf.groupby(part_field).size().rename("cell_count").reset_index()
    weight_sums = (
        cells_gdf.groupby(part_field)[weight_field]
        .sum()
        .rename("building_area_sum")
        .reset_index()
    )
    cell_area_sums = (
        cells_gdf.groupby(part_field)[cell_area_field]
        .sum()
        .rename(cell_area_field)
        .reset_index()
    )
    attrs = (
        cell_counts
        .merge(weight_sums, on=part_field, how="left")
        .merge(cell_area_sums, on=part_field, how="left")
    )

    # Dissolve geometry only, then attach explicitly summarized attributes.
    dissolved = cells_gdf[[part_field, "geometry"]].dissolve(
        by=part_field,
        as_index=False,
    )
    dissolved = dissolved.merge(attrs, on=part_field, how="left")

    dissolved["block_id"] = block_id
    dissolved["n_parts"] = n_parts
    dissolved["target_building_area"] = target_weight
    dissolved["share_of_total_building_area"] = dissolved["building_area_sum"] / total_weight
    dissolved["pct_of_target"] = dissolved["building_area_sum"] / target_weight
    dissolved["pct_deviation"] = (dissolved["building_area_sum"] - target_weight) / target_weight
    dissolved["partition_area_m2"] = dissolved.geometry.area
    dissolved["cell_vs_partition_area_diff_m2"] = (
        dissolved[cell_area_field] - dissolved["partition_area_m2"]
    )

    dissolved["solution_score"] = solution["score"]
    dissolved["boundary_score"] = solution["boundary_score"]
    dissolved["balance_penalty"] = solution["balance_penalty"]
    dissolved["internal_boundary_length"] = solution["boundary_length"]
    dissolved["diagonal_boundary_length"] = solution["diagonal_len"]
    dissolved["cut_edge_count"] = solution["cut_edges"]

    component_counts = {}
    for pid in sorted(cells_gdf[part_field].dropna().unique()):
        ids = cells_gdf.index[cells_gdf[part_field] == pid].tolist()
        component_counts[int(pid)] = component_count(graph, ids)

    dissolved["component_count"] = dissolved[part_field].astype(int).map(component_counts)
    dissolved["is_connected"] = dissolved["component_count"] == 1

    return dissolved


def solution_summary_rows(dissolved, part_field, block_id, block_folder_name, n_parts, output_gpkg, solution):
    """Build summary CSV rows."""
    rows = []
    for _, row in dissolved.iterrows():
        rows.append({
            "block_id": block_id,
            "block_folder": block_folder_name,
            "n_parts": n_parts,
            "partition_id": int(row[part_field]),
            "cell_count": int(row["cell_count"]),
            "building_area_sum": float(row["building_area_sum"]),
            "target_building_area": float(row["target_building_area"]),
            "share_of_total_building_area": float(row["share_of_total_building_area"]),
            "pct_of_target": float(row["pct_of_target"]),
            "pct_deviation": float(row["pct_deviation"]),
            "cell_area_m2": float(row[cell_area_field]),
            "partition_area_m2": float(row["partition_area_m2"]),
            "cell_vs_partition_area_diff_m2": float(row["cell_vs_partition_area_diff_m2"]),
            "component_count": int(row["component_count"]),
            "is_connected": bool(row["is_connected"]),
            "solution_score": float(solution["score"]),
            "boundary_score": float(solution["boundary_score"]),
            "balance_penalty": float(solution["balance_penalty"]),
            "internal_boundary_length": float(solution["boundary_length"]),
            "diagonal_boundary_length": float(solution["diagonal_len"]),
            "cut_edge_count": int(solution["cut_edges"]),
            "beam_width": BEAM_WIDTH,
            "max_split_candidates_per_step": MAX_SPLIT_CANDIDATES_PER_STEP,
            "output_gpkg": str(output_gpkg),
        })
    return rows



def load_weighted_tessellation(block_folder):
    """
    Read Step 3 tessellation geometry and join Step 4 building footprint area
    and tessellation-cell area by context_bldg_id.
    """
    tess_path = block_folder / tessellation_gpkg_name
    context_path = block_folder / context_gpkg_name

    if not tess_path.exists():
        raise FileNotFoundError(f"Missing Step 3 tessellation: {tess_path}")
    if not context_path.exists():
        raise FileNotFoundError(f"Missing Step 4 context dataset: {context_path}")

    tess = gpd.read_file(tess_path, layer=tessellation_layer)
    context = gpd.read_file(context_path, layer=context_layer)

    if context_id_field not in tess.columns:
        raise ValueError(
            f"Missing required field '{context_id_field}' in {tess_path} | {tessellation_layer}"
        )
    if context_id_field not in context.columns:
        raise ValueError(
            f"Missing required field '{context_id_field}' in {context_path} | {context_layer}"
        )
    for required_field in [weight_field, cell_area_field]:
        if required_field not in context.columns:
            raise ValueError(
                f"Missing required field '{required_field}' in "
                f"{context_path} | {context_layer}"
            )

    tess = tess.copy()
    context = context[
        [context_id_field, weight_field, cell_area_field]
    ].copy()

    tess[context_id_field] = pd.to_numeric(tess[context_id_field], errors="coerce")
    context[context_id_field] = pd.to_numeric(context[context_id_field], errors="coerce")
    context[weight_field] = pd.to_numeric(context[weight_field], errors="coerce")
    context[cell_area_field] = pd.to_numeric(
        context[cell_area_field],
        errors="coerce",
    )

    if tess[context_id_field].isna().any():
        raise ValueError(f"Null/non-numeric {context_id_field} values in {tess_path}")
    if context[context_id_field].isna().any():
        raise ValueError(f"Null/non-numeric {context_id_field} values in {context_path}")

    if context[context_id_field].duplicated().any():
        dup_count = int(context[context_id_field].duplicated(keep=False).sum())
        print(
            f"WARNING: Step 4 context data contains {dup_count:,} duplicate "
            f"{context_id_field} rows; keeping the first value per building."
        )
        context = context.drop_duplicates(subset=[context_id_field], keep="first")

    tess = tess.merge(context, on=context_id_field, how="left", validate="many_to_one")

    for required_field in [weight_field, cell_area_field]:
        missing_count = int(tess[required_field].isna().sum())
        if missing_count:
            raise ValueError(
                f"{missing_count:,} tessellation cell(s) did not receive "
                f"{required_field} from the Step 4 context dataset."
            )

    tess[weight_field] = tess[weight_field].fillna(0.0)
    tess[cell_area_field] = tess[cell_area_field].fillna(0.0)
    tess = tess.reset_index(drop=True).copy()
    tess = clean_geometry(tess)

    # QA only: Step 4 cell_area_m2 should match the Step 3 tessellation geometry.
    geom_cell_area = tess.geometry.area
    max_area_diff = float(
        (tess[cell_area_field] - geom_cell_area).abs().max()
    ) if len(tess) else 0.0
    if max_area_diff > 1e-6:
        print(
            "WARNING: maximum difference between Step 4 cell_area_m2 and "
            f"current tessellation geometry area is {max_area_diff:.9f} m2."
        )

    if "orig_cell_id" not in tess.columns:
        tess["orig_cell_id"] = tess.index + 1

    return tess


# =============================================================================
# MAIN
# =============================================================================

def main():
    summary_rows = []

    for block_folder_name in block_folders_to_process:
        block_id = block_folder_name.lstrip("_")
        block_folder = root_dir / block_folder_name
        output_gpkg = block_folder / output_gpkg_name

        print("\n" + "=" * 90)
        print(f"Processing block {block_folder_name}")
        print("=" * 90)

        try:
            gdf = load_weighted_tessellation(block_folder)
        except Exception as exc:
            print(f"FAILED to load weighted tessellation: {exc}")
            continue

        if output_gpkg.exists() and OVERWRITE_OUTPUT:
            try:
                output_gpkg.unlink()
                print(f"Deleted existing output: {output_gpkg}")
            except Exception as e:
                raise RuntimeError(
                    "Could not delete existing output GPKG. Close it in ArcGIS Pro and try again:\n"
                    f"{output_gpkg}\n\n{e}"
                )

        total_building_area = float(gdf[weight_field].sum())
        print(f"Total building footprint area: {total_building_area:,.2f}")
        print(f"Number of tessellation cells: {len(gdf):,}")

        print("Building adjacency graph and shared-boundary records...")
        graph, edge_records = build_adjacency_graph_and_edges(gdf)
        graph_components = nx.number_connected_components(graph)
        print(f"Adjacency graph components: {graph_components}")
        print(f"Adjacency edges: {len(edge_records):,}")

        if graph_components != 1:
            print(
                "WARNING: The input tessellation graph is not fully connected. "
                "Strictly connected final partitions may be impossible unless the input is repaired."
            )

        for n_parts in n_parts_to_process:
            print("\n" + "-" * 90)
            print(f"Searching for best {n_parts}-part partition with beam search")
            print("-" * 90)

            min_share, max_share = FINAL_SHARE_BOUNDS[n_parts]
            print(f"Allowed final share range per part: {min_share:.2f} to {max_share:.2f}")

            solution = find_best_partition_beam(gdf, graph, edge_records, n_parts)

            if solution is None:
                print(f"FAILED: no valid connected {n_parts}-part solution found under the share bounds.")
                continue

            part_field = f"part_n{n_parts}"
            cells_out = gdf.copy()
            cells_out[part_field] = assign_group_ids(cells_out, solution["groups"], part_field)
            cells_out["block_id"] = block_id
            cells_out["n_parts"] = n_parts
            cells_out["solution_score"] = solution["score"]
            cells_out["internal_boundary_length"] = solution["boundary_length"]
            cells_out["diagonal_boundary_length"] = solution["diagonal_len"]
            cells_out["cut_edge_count"] = solution["cut_edges"]

            dissolved = make_dissolved_layer(
                cells_gdf=cells_out,
                part_field=part_field,
                block_id=block_id,
                n_parts=n_parts,
                solution=solution,
                graph=graph,
            )

            if not dissolved["is_connected"].all():
                raise RuntimeError(
                    f"Internal error: disconnected final partition created for {block_folder_name}, n={n_parts}. "
                    "This should not happen because connectedness is enforced during search."
                )

            cells_layer = f"selected_cells_n{n_parts}"
            dissolved_layer = f"selected_dissolved_n{n_parts}"

            cells_out.to_file(output_gpkg, layer=cells_layer, driver="GPKG")
            dissolved.to_file(output_gpkg, layer=dissolved_layer, driver="GPKG")

            print(f"Wrote layer: {cells_layer}")
            print(f"Wrote layer: {dissolved_layer}")
            print(f"Solution score:             {solution['score']:,.2f}")
            print(f"Boundary score:             {solution['boundary_score']:,.2f}")
            print(f"Balance penalty:            {solution['balance_penalty']:,.2f}")
            print(f"Internal boundary length:   {solution['boundary_length']:,.2f}")
            print(f"Diagonal boundary length:   {solution['diagonal_len']:,.2f}")
            print(f"Cut edge count:             {solution['cut_edges']:,}")

            print("\nPartition balance:")
            print(
                dissolved[
                    [
                        part_field,
                        "cell_count",
                        "building_area_sum",
                        "cell_area_m2",
                        "share_of_total_building_area",
                        "pct_of_target",
                        "component_count",
                    ]
                ].to_string(index=False)
            )

            summary_rows.extend(
                solution_summary_rows(
                    dissolved=dissolved,
                    part_field=part_field,
                    block_id=block_id,
                    block_folder_name=block_folder_name,
                    n_parts=n_parts,
                    output_gpkg=output_gpkg,
                    solution=solution,
                )
            )

    summary_df = pd.DataFrame(summary_rows)

    if not summary_df.empty:
        summary_df.to_csv(summary_csv, index=False)
        print("\n" + "=" * 90)
        print(f"Partition summary CSV written to:\n{summary_csv}")
    else:
        print("\nNo BEAM partition outputs were created.")

    inventory_rows = []
    for block_folder_name in block_folders_to_process:
        parent_block = block_folder_name.lstrip("_")
        if summary_df.empty:
            successful_n = []
        else:
            mask = summary_df["block_folder"].astype(str) == str(block_folder_name)
            successful_n = sorted(
                {
                    int(v)
                    for v in summary_df.loc[mask, "n_parts"].dropna().tolist()
                }
            )

        inventory_rows.append(
            {
                "block_folder": block_folder_name,
                "parent_block": parent_block,
                "beam": 1,
                "n2_created": int(2 in successful_n),
                "n3_created": int(3 in successful_n),
                "n4_created": int(4 in successful_n),
                "status": "processed" if successful_n else "no_valid_solution_or_missing_input",
            }
        )

    pd.DataFrame(inventory_rows).to_csv(beam_inventory_csv, index=False)
    print(f"BEAM block inventory written to:\n{beam_inventory_csv}")




if __name__ == "__main__":
    main()